# Agent with Redis tool

### SETUP
Install Langchain dependencies

In [ ]:
%pip install -U langchain langchain-core langchain-community langchain-ollama redis --quiet

In [3]:
import redis
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

We will access a local Redis server running in Docker. Connect to the database and populate the database with some entries

In [5]:
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.set("user:001", "Name: Joe Bloggs, Role: Engineer")
r.set("user:002", "Name: Jane Ryan, Role: Designer")
r.set("product:001", "Name: Rope 1 meter, Price: $20")
r.set("product:002", "Name: Black hammer, Price: $30")

True

Instantiate the model

In [6]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Define 2 tools to read data from Redis

In [7]:
@tool
def get_user_data(user_id: str):
    """Retrieves user data from Redis given a user ID."""
    return r.get(f"user:{user_id}")

@tool
def get_product_data(product_id: str):
    """Retrieves product data from Redis given a product ID."""
    return r.get(f"product:{product_id}")

Give the agent "memory"

In [8]:
checkpointer = InMemorySaver()

Create the agent

In [9]:
SYSTEM_PROMPT = """You are a helpful assistant that interacts with a Redis database.
You have access to a tool:
- get_user_data: use this to get data about a specific user
- get_product_data: use this to get data about a specific product"""

agent = create_agent(
    model=llm,
    tools=[get_user_data, get_product_data],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer
)

### TESTING
Let's try it out

In [10]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the role of user 001 and the price of product 002?"}]},
    config={"configurable": {"thread_id": "00001"}}
)

In [11]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001 and the price of product 002?', additional_kwargs={}, response_metadata={}, id='373e2990-92e8-4361-aaae-8f5fcdcf6eaa'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T00:49:38.430193Z', 'done': True, 'done_reason': 'stop', 'total_duration': 35360190000, 'load_duration': 8286623100, 'prompt_eval_count': 197, 'prompt_eval_duration': 153127300, 'eval_count': 344, 'eval_duration': 26489560500, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e23f5-5c5b-7f02-a376-0188429e1a18-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '4d69d949-9f0f-4f7c-b404-96e42cae4079', 'type': 'tool_call'}, {'name': 'get_product_data', 'args': {'product_id': '002'}, 'id': '2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_toke

In [12]:
for m in response["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

What is the role of user 001 and the price of product 002?
None
================================== Ai Message ==================================
Tool Calls:
  get_user_data (4d69d949-9f0f-4f7c-b404-96e42cae4079)
 Call ID: 4d69d949-9f0f-4f7c-b404-96e42cae4079
  Args:
    user_id: 001
  get_product_data (2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd)
 Call ID: 2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd
  Args:
    product_id: 002
None
================================= Tool Message =================================
Name: get_user_data

Name: Joe Bloggs, Role: Engineer
None
================================= Tool Message =================================
Name: get_product_data

Name: Black hammer, Price: $30
None
================================== Ai Message ==================================

The role of user 001 is Engineer, and the price of product 002 is $30.
None


In [13]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "I have $70, how many hammers can I buy?"}]},
    config={"configurable": {"thread_id": "00001"}}
)

In [14]:
for m in response["messages"]:
    print(m.pretty_print())

================================ Human Message =================================

What is the role of user 001 and the price of product 002?
None
================================== Ai Message ==================================
Tool Calls:
  get_user_data (4d69d949-9f0f-4f7c-b404-96e42cae4079)
 Call ID: 4d69d949-9f0f-4f7c-b404-96e42cae4079
  Args:
    user_id: 001
  get_product_data (2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd)
 Call ID: 2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd
  Args:
    product_id: 002
None
================================= Tool Message =================================
Name: get_user_data

Name: Joe Bloggs, Role: Engineer
None
================================= Tool Message =================================
Name: get_product_data

Name: Black hammer, Price: $30
None
================================== Ai Message ==================================

The role of user 001 is Engineer, and the price of product 002 is $30.
None
================================ Human Message ==========

In [17]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "I want to gift some rope to an engineer. Who can I give it to and how many meters of rope can I buy?"}]},
    config={"configurable": {"thread_id": "00001"}}
)

In [19]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001 and the price of product 002?', additional_kwargs={}, response_metadata={}, id='373e2990-92e8-4361-aaae-8f5fcdcf6eaa'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T00:49:38.430193Z', 'done': True, 'done_reason': 'stop', 'total_duration': 35360190000, 'load_duration': 8286623100, 'prompt_eval_count': 197, 'prompt_eval_duration': 153127300, 'eval_count': 344, 'eval_duration': 26489560500, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e23f5-5c5b-7f02-a376-0188429e1a18-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '4d69d949-9f0f-4f7c-b404-96e42cae4079', 'type': 'tool_call'}, {'name': 'get_product_data', 'args': {'product_id': '002'}, 'id': '2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_toke

In [20]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "I believe the product ID is 001?"}]},
    config={"configurable": {"thread_id": "00001"}}
)

In [21]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001 and the price of product 002?', additional_kwargs={}, response_metadata={}, id='373e2990-92e8-4361-aaae-8f5fcdcf6eaa'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T00:49:38.430193Z', 'done': True, 'done_reason': 'stop', 'total_duration': 35360190000, 'load_duration': 8286623100, 'prompt_eval_count': 197, 'prompt_eval_duration': 153127300, 'eval_count': 344, 'eval_duration': 26489560500, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e23f5-5c5b-7f02-a376-0188429e1a18-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '4d69d949-9f0f-4f7c-b404-96e42cae4079', 'type': 'tool_call'}, {'name': 'get_product_data', 'args': {'product_id': '002'}, 'id': '2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_toke

In [23]:
for m in response["messages"][-4:]:
    print(m.pretty_print())

================================ Human Message =================================

I believe the product ID is 001?
None
================================== Ai Message ==================================
Tool Calls:
  get_product_data (747e9f1c-b59d-4a4e-8715-14cd0a4b3b00)
 Call ID: 747e9f1c-b59d-4a4e-8715-14cd0a4b3b00
  Args:
    product_id: 001
None
================================= Tool Message =================================
Name: get_product_data

Name: Rope 1 meter, Price: $20
None
================================== Ai Message ==================================

With $70, you can buy **3 meters** of rope.
None


In [25]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "How much money will I have left?"}]},
    config={"configurable": {"thread_id": "00001"}})

In [26]:
import pprint
pprint.pprint(response)

{'messages': [HumanMessage(content='What is the role of user 001 and the price of product 002?', additional_kwargs={}, response_metadata={}, id='373e2990-92e8-4361-aaae-8f5fcdcf6eaa'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4:e4b', 'created_at': '2026-05-14T00:49:38.430193Z', 'done': True, 'done_reason': 'stop', 'total_duration': 35360190000, 'load_duration': 8286623100, 'prompt_eval_count': 197, 'prompt_eval_duration': 153127300, 'eval_count': 344, 'eval_duration': 26489560500, 'logprobs': None, 'model_name': 'gemma4:e4b', 'model_provider': 'ollama'}, id='lc_run--019e23f5-5c5b-7f02-a376-0188429e1a18-0', tool_calls=[{'name': 'get_user_data', 'args': {'user_id': '001'}, 'id': '4d69d949-9f0f-4f7c-b404-96e42cae4079', 'type': 'tool_call'}, {'name': 'get_product_data', 'args': {'product_id': '002'}, 'id': '2e5ea9dd-66ac-4d0c-b01b-84a6d0c209fd', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 197, 'output_toke